In [ ]:
!pip install pandas numpy matplotlib seaborn scipy tqdm beautifulsoup4 requests
!pip3 install folium


In [ ]:

# 항상 import하는 라이브러리 들
import pandas as pd
import numpy as np
import re, requests, json
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import scipy.stats as stats
from scipy.optimize import curve_fit

import random

import matplotlib.pyplot as plt

import glob

import json

from datetime import datetime

import time 

import os
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
import requests
import urllib.request

In [ ]:
# 본래 크롤링 하려고 했으나, 파일 그냥 다운받아서 써도 무방하다 하셔서 주석처리

""""
html = response.text

# BeautifulSoup을 사용하여 HTML 파싱
soup = BeautifulSoup(html, "html.parser")

# 다운로드 링크 목록 추출
download_links = soup.find_all("a", class_="download")

# 최신 날짜 데이터 찾기
latest_date = ""
target_zip_file = None
for link in download_links:
    href = link["href"]
    if "DONG_" in href:
        date = href.split("DONG_")[1][:6]
        if date > latest_date:
            latest_date = date
            target_zip_file = href

# 다운로드 링크 출력
if target_zip_file is not None:
    print(target_zip_file)
    urllib.request.urlretrieve(target_zip_file, "다운로드할_파일명.zip")
    print("파일 다운로드 완료")
else:
    print("최신 날짜 데이터를 찾을 수 없습니다.")
"""

In [ ]:
# 데이터 프레임 읽는 부분
import pandas as pd
#(csv파일 다운받은 경로 입력력)
file_path = "/home/addd/Downloads/INTERN_YUSEONGMIN/EDA/LOCAL_PEOPLE_DONG_202311.csv"

# CSV 파일을 데이터프레임으로 읽어오기
df = pd.read_csv(file_path)

# 데이터프레임 출력
df

In [ ]:
#df를 가져오면 이상하게 꼭 제일 왼쪽에 있는 열을 인덱스로 삼아서 오류 남.
# 열의 값들을 한 칸씩 밀기
df[df.columns] = df[df.columns].shift(1, axis=1)

# 행의 인덱스 번호를 '기준일ID' 열에 넣기
df['기준일ID'] = df.index

# 인덱스 번호를 0부터 행의 개수로 재설정
df = df.reset_index(drop=True)

# 데이터 프레임 NaN값을 0으로 변환
df = df.fillna(0)

In [ ]:
df

In [ ]:
import pandas as pd
# 법정동 코드 정보 다운로드(csv 다운 받은 경로 입력)
file_path = "/home/addd/Downloads/INTERN_YUSEONGMIN/EDA/df_hangjungdong_code_data_2024.csv"

# CSV 파일을 데이터프레임으로 읽어오기

df_1 = pd.read_csv(file_path)

df_1

In [ ]:
# df_1_mangwoo3 데이터프레임 복사
df_1_copy = df_1.copy()

# 시군구코드+(0이 아닌)행정동코드
df_1_copy['시군구_행정동'] = df_1_copy.apply(lambda row: str(row['시군구코드']) + str(row['행정동코드']) if row['행정동코드'] != 0 else '', axis=1)

# '행정동코드' 컬럼의 데이터 타입을 문자열로 변환
df['행정동코드'] = df['행정동코드'].astype(str)

# 병합 시도
merged_df = pd.merge(df, df_1_copy, left_on='행정동코드', right_on='시군구_행정동', how='inner')

# '시군구_행정동'과 '기준일ID'가 동일한 행 제거(안 하면 df보다 더 많은 행이 생성됨)
df_unique = merged_df.drop_duplicates(subset=['시군구_행정동', '기준일ID','총생활인구수'])
df_unique 


In [ ]:
#주소 데이터 이미 구해서 중복 막으려고 주석처리(만약, 새로운 데이터를 추가해야 하면 사용)
'''from geopy.geocoders import Nominatim
#주소에 따른 위도, 경도 데이터를 수집(이상하게 오래걸리면 오류가 발생할 수 있으니 tqdm사용)
# Nominatim 객체를 생성합니다.
geolocator = Nominatim(user_agent="geExercises")

# df_1의 각 행에 대해 geocode 함수를 적용합니다.
df_1['lat'] = None
df_1['lng'] = None

# 경기도 구리시의 위도와 경도를 구합니다.
default_location = geolocator.geocode("경기도 구리시 수택동")
default_lat = default_location.latitude
default_lng = default_location.longitude

def do_geocode(address, attempt=1, max_attempts=5):
    try:
        return geolocator.geocode(address)
    except GeocoderTimedOut:
        if attempt <= max_attempts:
            return do_geocode(address, attempt=attempt+1)
        raise
count=0
time.sleep(1)
for i in tqdm(df_1.index):
    # 주소를 생성합니다.
    address = str(df_1.loc[i, '시도명']) + ' ' + str(df_1.loc[i, '시군구명']) + ' ' + str(df_1.loc[i, '법정동명']) + ' ' + str(df_1.loc[i, '행정동명'])
    # Geocoding 요청을 보냅니다.
    location = do_geocode(address)
    
    # 결과에서 위도와 경도를 가져와 df_1에 저장합니다.
    if location:
        df_1.loc[i, 'lat'] = location.latitude
        df_1.loc[i, 'lng'] = location.longitude
    else:  # 결과가 없을 경우, 경기도 구리시의 위도와 경도를 저장합니다.
        df_1.loc[i, 'lat'] = default_lat
        df_1.loc[i, 'lng'] = default_lng
    time.sleep(0.4)  # Rate limiting을 준수하기 위해 1초 지연
df_1.to_csv('df_hangjungdong_code_data_2024.csv', index=False)
'''

In [ ]:
'''# 성별 나눠서 5세 단위로 히트맵 생성하는 함수
import folium
from folium.plugins import HeatMap

# 히트맵 생성 함수
def create_heatmap(df, column_name):
    # 지도의 중심을 지정하기 위해 위도와 경도의 평균값을 구합니다.
    average_lat = df['lat'].mean()
    average_lng = df['lng'].mean()

    # 지도를 생성합니다.
    m = folium.Map(location=[average_lat, average_lng], zoom_start=13)
    
    # 히트맵 데이터 만들기 (위도, 경도, 해당 열의 값)
    heat_data = [[row['lat'], row['lng'], row[column_name]] for index, row in df.iterrows()]

    # 히트맵 추가, radius 파라미터로 크기 조절, min_opacity로 투명도 조절
    HeatMap(heat_data, radius=30, min_opacity=0.5).add_to(m)  # 반경을 30으로, 최소 투명도를 0.5로 설정

    # 지도 저장
    m.save(f'heatmap_{column_name}.html')

# 성별 및 연령별로 히트맵 생성 및 저장
age_gender_columns = ['남자0세부터9세생활인구수', '남자10세부터14세생활인구수', '남자15세부터19세생활인구수', '남자20세부터24세생활인구수',
                      '남자25세부터29세생활인구수', '남자30세부터34세생활인구수', '남자35세부터39세생활인구수', '남자40세부터44세생활인구수',
                      '남자45세부터49세생활인구수', '남자50세부터54세생활인구수', '남자55세부터59세생활인구수', '남자60세부터64세생활인구수',
                      '남자65세부터69세생활인구수', '남자70세이상생활인구수', '여자0세부터9세생활인구수', '여자10세부터14세생활인구수',
                      '여자15세부터19세생활인구수', '여자20세부터24세생활인구수', '여자25세부터29세생활인구수', '여자30세부터34세생활인구수',
                      '여자35세부터39세생활인구수', '여자40세부터44세생활인구수', '여자45세부터49세생활인구수', '여자50세부터54세생활인구수',
                      '여자55세부터59세생활인구수', '여자60세부터64세생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']

for column in tqdm(age_gender_columns):
    create_heatmap(df_unique, column)
'''

In [18]:

import folium
from folium.plugins import HeatMap
# 히트맵 생성 함수
def create_heatmap_detail(merged, start_age, end_age, hour, gender, exclude_coords=None):
    # 해당 시간대에 해당하는 데이터만 필터링합니다.
    filtered_df = merged[merged['시간대구분'] == hour]
    
    # 성별과 나이 범위에 따라 필터링합니다.
    if start_age == 70:
        age_gender_column = f'{gender}{start_age}세이상생활인구수'
    else:
        age_gender_column = f'{gender}{start_age}세부터{end_age}세생활인구수'

    filtered_df = filtered_df[(filtered_df[age_gender_column] > 0)]

    # 지도의 중심을 지정하기 위해 위도와 경도의 평균값을 구합니다.
    average_lat = filtered_df['lat'].mean()
    average_lng = filtered_df['lng'].mean()
    
    # 구리시 좌표를 제외할 경우 필터링 로직을 적용합니다.
    if exclude_coords:
        tolerance = 1e-3  # 예: 0.001의 허용 오차
        lat, lng = exclude_coords
        filtered_df = filtered_df[
            ~(
                np.isclose(filtered_df['lat'], lat, atol=tolerance) &
                np.isclose(filtered_df['lng'], lng, atol=tolerance)
            )
        ]

    # 지도를 생성합니다.
    m = folium.Map(location=[average_lat, average_lng], zoom_start=13)
    
    # 히트맵 데이터 만들기 (위도, 경도, 인구수)
    heat_data = [
        [row['lat'], row['lng'], row[age_gender_column]]
        for index, row in filtered_df.iterrows()
    ]

    # 히트맵 추가, radius 파라미터로 크기 조절, min_opacity로 투명도 조절
    HeatMap(heat_data, radius=30, min_opacity=0.5).add_to(m)

    # 지도 저장
    

    if start_age == 70:
        m.save(f'/home/addd/Downloads/heatmap_{hour}_{gender}{start_age}세이상.html')
    else:
        m.save(f'/home/addd/Downloads/heatmap_{hour}_{gender}{start_age}세부터{end_age}세.html')


# 구리시 좌표 설정(이전에 비어있던 값들 제거)
exclude_coords = (37.5962736, 127.1419836)  # 구리시의 대략적인 좌표
 
# 특정 시간대와 나이 범위, 성별에 대한 히트맵 생성 및 저장
hour = 7  # 시간대
start_age = 70  # 시작 나이
end_age = '무의미' # 종료 나이
gender = '남자'  # 성별

create_heatmap_detail(df_unique, start_age, end_age, hour, gender, exclude_coords=exclude_coords)

In [ ]:
# csv파일로 저장하기기
df_unique.to_csv('unique_df.csv', index=False)